# Base-rate merged results

Explore `data/base_rate/base_rate_merged_results.csv` from a benchmark run.

Each row has **`score`** (`true`/`false`): whether the parsed answer matches **`scepticism_score_target`**. Unparseable rows have `score=false` and `parseable=false`.

In [2]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "base_rate").is_dir():
    ROOT = ROOT.parent

MERGED_DIR = ROOT / "data" / "base_rate"
MERGED_CSV = None
for name in (
    "base_rate_merged_results.csv",
    "base_rate_merged_results (2).csv",
):
    candidate = MERGED_DIR / name
    if candidate.is_file():
        MERGED_CSV = candidate
        break
if MERGED_CSV is None:
    raise FileNotFoundError(
        f"Missing merged results under {MERGED_DIR}. Run the base-rate benchmark first "
        "(benchmark/base-rate-benchmark.ipynb)."
    )

df = pd.read_csv(MERGED_CSV)

if "score" in df.columns:
    df["score_value"] = df["score"].astype(str).str.lower().eq("true").astype(int)
elif "score_outcome" in df.columns:
    df["score_value"] = (df["score_outcome"] == "normative").astype(int)
else:
    raise KeyError("Merged CSV must include 'score' or legacy 'score_outcome'.")

if "parseable" in df.columns:
    df["parseable_bool"] = df["parseable"].astype(str).str.lower().eq("true")

print("Loaded:", MERGED_CSV)
print("Rows:", len(df))
print("Models:", sorted(df["model"].unique()))
print("Vignettes:", df["vignette_name"].nunique())
df.head()

Loaded: c:\src2\sceptical-llms\data\base_rate\base_rate_merged_results.csv
Rows: 18
Models: ['google/gemini-2.5-flash']
Vignettes: 3


,example_id,vignette_name,problem_type,intersection_size,response_type,has_statistics,variant,prompt,well_posed,normative,...,parsed_choice,parsed_confidence,scoring_type,parseable,score_outcome,normative.1,biased,lure_matched,score_value,parseable_bool
0,actor_waiter_overlap__overlap__mc_full_no_probs,actor waiter overlap,overlap,small,mc_full,False,mc_full_no_probs,You are a statistical consultant. Your task is...,False,False,...,D,3,mc_full,True,biased,False,True,q_d*s_d only (second pathway; assumes partition),0,True
1,actor_waiter_overlap__overlap__mc_full_probs,actor waiter overlap,overlap,small,mc_full,True,mc_full_probs,You are a statistical consultant. Your task is...,False,False,...,D,5,mc_full,True,biased,False,True,partition shortcut (assumes P(C∩D|A)=0),0,True
2,actor_waiter_overlap__overlap__mc_numeric_no_p...,actor waiter overlap,overlap,small,mc_numeric,False,mc_numeric_no_probs,You are a statistical consultant. Your task is...,False,False,...,D,4,mc_numeric,True,biased,False,True,q_d*s_d only (second pathway; assumes partition),0,True
3,actor_waiter_overlap__overlap__mc_numeric_probs,actor waiter overlap,overlap,small,mc_numeric,True,mc_numeric_probs,You are a statistical consultant. Your task is...,False,False,...,E,5,mc_numeric,True,biased,False,True,q_c*s_c × q_d*s_d (product; assumes independence),0,True
4,actor_waiter_overlap__overlap__open_no_probs,actor waiter overlap,overlap,small,open,False,open_no_probs,You are a statistical consultant. Your task is...,False,False,...,NaN,4,open,True,off_target,False,False,NaN,0,True


In [3]:
df.columns

Index(['example_id', 'vignette_name', 'problem_type', 'intersection_size',
       'response_type', 'has_statistics', 'variant', 'prompt', 'well_posed',
       'normative', 'p_c_and_d_given_a', 'normative_choice',
       'normative_percent', 'normative_open', 'confidence_required',
       'numeric_score_percent', 'numeric_score_choice', 'scepticism_required',
       'scepticism_score_target', 'option_a_label', 'option_b_label',
       'option_c_label', 'option_d_label', 'option_e_label', 'option_a_lure',
       'option_b_lure', 'option_c_lure', 'option_d_lure', 'option_e_lure',
       'option_f_label', 'option_g_label', 'option_h_label', 'option_f_lure',
       'option_g_lure', 'option_h_lure', 'model', 'llm_response', 'reasoning',
       'answer_line', 'confidence_line', 'parsed_answer_type',
       'parsed_percent', 'parsed_choice', 'parsed_confidence', 'scoring_type',
       'parseable', 'score_outcome', 'normative.1', 'biased', 'lure_matched',
       'score_value', 'parseable_bool']

In [4]:
df['reasoning'].value_counts(), df['confidence_required'].value_counts(), df['normative_choice'].value_counts()

(Series([], Name: count, dtype: int64),
 confidence_required
 True    18
 Name: count, dtype: int64,
 normative_choice
 B    8
 D    3
 C    1
 Name: count, dtype: int64)

## Scores by `response_type`

In [5]:
RESPONSE_TYPE_ORDER = ["open", "mc_numeric", "mc_full"]


def score_summary_table(group_col: str, *, order: list[str] | None = None) -> pd.DataFrame:
    """Counts, parseability mix, and mean score for each group value."""
    work = df.copy()
    if "parseable_bool" not in work.columns:
        work["parseable_bool"] = True
    work["score_miss"] = work["parseable_bool"] & (work["score_value"] == 0)
    work["unparseable_row"] = ~work["parseable_bool"]

    grouped = work.groupby(group_col, observed=True)
    summary = pd.DataFrame(
        {
            "n": grouped.size(),
            "score_true": grouped["score_value"].sum(),
            "score_false": grouped["score_miss"].sum(),
            "unparseable": grouped["unparseable_row"].sum(),
            "score_rate": grouped["score_value"].mean(),
        }
    )
    summary["score_pct"] = (summary["score_rate"] * 100).round(1)

    if order:
        summary = summary.reindex([value for value in order if value in summary.index])

    return summary


by_response_type = score_summary_table("response_type", order=RESPONSE_TYPE_ORDER)
by_response_type

,n,score_true,score_false,unparseable,score_rate,score_pct
response_type,,,,,,
open,6,1,5,0,0.166667,16.7
mc_numeric,6,1,5,0,0.166667,16.7
mc_full,6,1,5,0,0.166667,16.7


## Scores by `variant`

In [6]:
VARIANT_ORDER = [
    "open_probs",
    "open_no_probs",
    "mc_numeric_probs",
    "mc_numeric_no_probs",
    "mc_full_probs",
    "mc_full_no_probs",
]

by_variant = score_summary_table("variant", order=VARIANT_ORDER)
by_variant

,n,score_true,score_false,unparseable,score_rate,score_pct
variant,,,,,,
open_probs,3,1,2,0,0.333333,33.3
open_no_probs,3,0,3,0,0.000000,0.0
mc_numeric_probs,3,1,2,0,0.333333,33.3
mc_numeric_no_probs,3,0,3,0,0.000000,0.0
mc_full_probs,3,1,2,0,0.333333,33.3
mc_full_no_probs,3,0,3,0,0.000000,0.0


## Scores by `vignette_name`

In [7]:
by_vignette = score_summary_table(
    "vignette_name",
    order=sorted(df["vignette_name"].unique()),
)
by_vignette

,n,score_true,score_false,unparseable,score_rate,score_pct
vignette_name,,,,,,
CA Trump voter,6,3,3,0,0.5,50.0
actor waiter overlap,6,0,6,0,0.0,0.0
college STEM work,6,0,6,0,0.0,0.0


## Optional: split by model when multiple LLMs are present

In [8]:
if df["model"].nunique() > 1:
    display(
        df.groupby(["model", "response_type"], observed=True)["score_value"]
        .mean()
        .unstack("response_type")
        .reindex(columns=RESPONSE_TYPE_ORDER)
        .round(3)
    )
    display(
        df.groupby(["model", "variant"], observed=True)["score_value"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
else:
    print("Single model in file — see tables above.")

Single model in file — see tables above.
